#### Please Run in colab

# Setup

### Environment Setup

In [1]:
!pip install instructor

In [2]:
# !cp /content/macro_financial_forecasting/applications/macro_financial_forecasting/data/danidanou_Bloomberg_Financial_News_train /content/
!cd /content
!rm -rf macro_financial_forecasting

In [3]:
!git clone https://github.com/chuanbinp/macro_financial_forecasting.git

Cloning into 'macro_financial_forecasting'...
remote: Enumerating objects: 1867, done.
remote: Counting objects: 100% (683/683), done.
remote: Compressing objects: 100% (233/233), done.
remote: Total 1867 (delta 516), reused 471 (delta 449), pack-reused 1184 (from 2)
Receiving objects: 100% (1867/1867), 39.55 MiB | 17.85 MiB/s, done.
Resolving deltas: 100% (1174/1174), done.


In [4]:
# !cp /content/danidanou_Bloomberg_Financial_News_train /content/macro_financial_forecasting/applications/macro_financial_forecasting/data/danidanou_Bloomberg_Financial_News_train

In [5]:
%cd macro_financial_forecasting/applications/macro_financial_forecasting/src

/content/macro_financial_forecasting/applications/macro_financial_forecasting/src


### Mount GDrive

In [12]:
from google.colab import drive

# This will prompt you to authorize Colab to access your Google Drive.
drive.mount('/content/gdrive')
GDRIVE_PATH = "/content/gdrive/MyDrive/macro_financial_forecasting_files/"

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


### Code Setup

In [7]:
from config import Config
from train_data_loader import TrainDataLoader
from data_model.bloomberg_news_entry import BloombergNewsEntry
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
config = Config("../config.env")

train_data_loader = TrainDataLoader(config)

# Train Data Loader

In [8]:
print("Starting loading pipeline ...")
print(f"Config: {config}")

train_ds = train_data_loader.load()
print("Loading pipeline completed.")

Starting loading pipeline ...
Config: Config(
  gemini_api_key: !secret!
  openai_api_key: !secret!
  llm_model: openai/gpt-5-nano-2025-08-07
  industries: ['Information Technology', 'Health Care', 'Financials', 'Consumer Discretionary', 'Communication Services', 'Industrials', 'Consumer Staples', 'Energy', 'Utilities', 'Real Estate', 'Materials', 'General Market', 'None']
  dataset_name: danidanou/Bloomberg_Financial_News
  dataset_dir: ../data/
  rss_feeds: ['https://feeds.bloomberg.com/news/news.rss', 'https://feeds.bloomberg.com/markets/news.rss', 'https://feeds.bloomberg.com/business/news.rss', 'https://feeds.bloomberg.com/technology/news.rss', 'https://feeds.bloomberg.com/politics/news.rss', 'https://feeds.bloomberg.com/wealth/news.rss', 'https://feeds.bloomberg.com/economics/news.rss', 'https://feeds.bloomberg.com/green/news.rss', 'https://feeds.bloomberg.com/pursuits/news.rss', 'https://feeds.bloomberg.com/opinion/news.rss', 'https://feeds.bloomberg.com/finance/news.rss', 'http

bloomberg_financial_data.parquet.gzip:   0%|          | 0.00/482M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/446762 [00:00<?, ? examples/s]


--- Download Successful! ---


Map:   0%|          | 0/446762 [00:00<?, ? examples/s]

Training dataset processed.

--- Starting validation of 446762 entries ---


Validating entries: 100%|██████████| 446762/446762 [00:30<00:00, 14626.23it/s]


--- Validation Complete! ---
Training dataset validated.
Saving processed dataset to local cache at '../data/danidanou_Bloomberg_Financial_News_train'...
Total number of rows: 446762
Loading pipeline completed.


# Data Processing Pipeline

Update these 2 variable to specify which indices to process.

In [9]:
DATA_START=0
DATA_END=100

In [17]:
from processor import NewsProcessor
import nest_asyncio
nest_asyncio.apply()

processor = NewsProcessor(config)
train_ds = processor.remove_redundant_info(train_ds[DATA_START:DATA_END])
df = processor.enrich_news_entries_with_classifications(train_ds, save_path=f"{GDRIVE_PATH}processed_news") #Sample size
df = processor.group_by_date_and_industry(df, save_path=f"{GDRIVE_PATH}grouped_news")
df = processor.filter_and_analyze_news(df)
df = processor.extract_impactful_news(df, top_n=3, save_path=f"{GDRIVE_PATH}impact_news")
df = processor.get_consolidated_sentiment(df, save_path=f"{GDRIVE_PATH}sentiment_news")

Device set to use cuda:0


Processing 100 news entries...


Industry Classification: 100%|██████████| 4/4 [00:04<00:00,  1.16s/batch]


Completed processing 100 entries

Dropped 2 (Industry, Date) pairs with Industry='None'
Remaining pairs: 19

Summary Statistics:
Total unique (Industry, Date) pairs: 19
Average articles per pair: 5.11
Max articles in a pair: 51
Min articles in a pair: 1
25th percentile: 1.0
50th percentile: 2.0
75th percentile: 4.5
Number of pairs with at least 3 articles: 6
Total articles: 97


Extracting top 3 impactful news per (Industry, Date) pair...


Processing groups: 100%|██████████| 19/19 [00:00<00:00, 40576.26it/s]


Processing 19 news entries...


FinBERT Sentiment: 100%|██████████| 1/1 [00:00<00:00,  7.41batch/s]

Completed processing 19 entries


In [18]:
df = await processor.get_explanation(df, save_path=f"{GDRIVE_PATH}sentiment_news")

Explanation: 100%|██████████| 19/19 [00:23<00:00,  1.22s/it]


In [19]:
df

,Industry,Date,News,ArticleCount,ImpactfulNews,AvgSentimentScore,SentimentScore,SentimentExplanation
0,Communication Services,2011-10-06,[{'Headline': 'FCC to Revamp Phone Subsidy to ...,2,[{'Headline': 'Euro-Area Leaders to Hold Summi...,0.709191,-0.291917,Overall sentiment for the Communications Servi...
1,Consumer Discretionary,2011-10-06,[{'Headline': 'PepsiCo May Purchase Russian Dr...,1,[{'Headline': 'PepsiCo May Purchase Russian Dr...,0.881740,0.888237,Explanation: The PepsiCo news item about poten...
2,Consumer Staples,2011-10-06,[{'Headline': 'Ukraine’s Grain Harvest Advance...,1,[{'Headline': 'Ukraine’s Grain Harvest Advance...,-0.918589,-0.917441,The article triggers a strong negative sentime...
3,Energy,2011-10-06,[{'Headline': 'Clean-Tech Companies Should Get...,9,[{'Headline': 'Norway Boosts Mongstad Carbon-S...,0.252093,-0.252850,The FinBERT score is mildly negative (-0.253)....
4,Financials,2011-10-06,[{'Headline': 'Ivory Coast Keeps Cocoa Export ...,51,[{'Headline': 'Remittances to Vietnam Thru Jul...,-0.306989,-0.032282,The FinBERT score of -0.032 reflects a largely...
5,General Market,2011-10-06,[{'Headline': 'Farmland Seen Returning Up to 1...,4,[{'Headline': 'GE Study Finds Recession’s Job ...,-0.256293,-0.873807,The combined news is notably negative for the ...
6,Health Care,2011-10-06,[{'Headline': 'House Panel Seeks Details on IR...,2,[{'Headline': 'Emdeon Said to Set Rate on $1.2...,0.666714,0.682417,The FinBERT score of 0.682 indicates a positiv...
7,Industrials,2011-10-06,[{'Headline': 'Airbus German Workers Plan Work...,5,"[{'Headline': 'Polish Stocks: Getin, KGHM, Lot...",-0.296801,-0.868415,The overall sentiment score for the Industrial...
8,Information Technology,2011-10-06,[{'Headline': 'Fans Hold IPhone-Lit Vigils for...,2,[{'Headline': 'Fans Hold IPhone-Lit Vigils for...,0.412332,0.002113,Aggregate sentiment for Information Technology...
9,Materials,2011-10-06,[{'Headline': 'USDA Boxed Beef Cutout Closing ...,5,[{'Headline': 'Ukraine September Consumer Pric...,0.853697,0.875244,The Materials signal is bullish. Positive pric...
